In [1]:
import operator
from typing import TypedDict, Annotated, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
import json
from dotenv import load_dotenv
import os

load_dotenv(".env")
api_key = os.getenv("GROQ_TOKEN")

# Define the Global State
class ResearchState(TypedDict):
    original_query: str
    search_queries: List[str]
    # Annotated with operator.add means we append to the list on each loop, not overwrite
    gathered_chunks: Annotated[List[Dict[str, Any]], operator.add] 
    failed_urls: Annotated[List[str], operator.add]
    missing_information: List[str] # If this has items, the graph will loop
    draft_report: str
    is_complete: bool

In [2]:
# Initialize your LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0,api_key=api_key)

def supervisor_agent(state: ResearchState):
    """Analyzes the query or missing information and decides what to search for next."""
    print("---SUPERVISOR AGENT: Generating Queries---")
    
    query = state["original_query"]
    missing = state.get("missing_information", [])
    
    # If looping back because of missing info, focus ONLY on the missing info
    if missing:
        prompt = f"We are missing: {missing}. Generate 2 highly specific search queries to find this."
    else:
        prompt = f"Original goal: {query}. Generate 3 initial academic search queries."
        
    # [Insert LLM call here to generate queries]
    # For skeleton purposes, let's pretend the LLM returned these:
    new_queries = ["thermal mass of bamboo", "specific heat capacity bamboo architecture"]
    
    # Return the updated state
    return {"search_queries": new_queries}


def web_and_ingestion_agent(state: ResearchState):
    """Executes searches, scrapes, chunks (LlamaIndex), and stores in Vector DB."""
    print("---INGESTION AGENT: Searching and RAG processing---")
    queries = state["search_queries"]
    
    # [Insert Tavily/ArXiv Tool Calling Here]
    # [Insert LlamaIndex Semantic Chunking & AWS RDS Vector Storage Here]
    # [Insert Hybrid Retrieval & Re-ranking Here]
    
    # Let's pretend we retrieved these chunks:
    retrieved_chunks = [{"content": "Bamboo has a specific heat capacity of...", "source": "arxiv"}]
    failed = ["badwebsite.com/bamboo"]
    
    # We append the new chunks to the state
    return {"gathered_chunks": retrieved_chunks, "failed_urls": failed}


def synthesis_and_critique_agent(state: ResearchState):
    """Drafts the report and grades its own work."""
    print("---SYNTHESIS AGENT: Drafting and Critiquing---")
    
    chunks = state["gathered_chunks"]
    original_query = state["original_query"]
    
    # [Insert LLM call to draft report based on chunks]
    draft = "Here is the report on earth blocks... but I couldn't find much on bamboo."
    
    # [Insert LLM call to critique the draft against the original query]
    # The LLM outputs JSON indicating if info is missing.
    # Pretend the LLM realized it's missing structural data:
    
    critique_result = {
        "status": "incomplete",
        "missing_information": ["compressive strength of structural bamboo"]
    }
    
    if critique_result["status"] == "incomplete":
        return {
            "draft_report": draft,
            "missing_information": critique_result["missing_information"],
            "is_complete": False
        }
    else:
        return {
            "draft_report": draft,
            "missing_information": [],
            "is_complete": True
        }

In [3]:
def routing_logic(state: ResearchState) -> str:
    """Decides if the graph should loop back or finish."""
    if state.get("is_complete", False):
        print("-> Routing: Data is complete. Ending graph.")
        return "end"
    else:
        print(f"-> Routing: Missing {state['missing_information']}. Looping back to Supervisor.")
        return "continue"

In [4]:
# 1. Initialize the Graph with your State schema
workflow = StateGraph(ResearchState)

# 2. Add the nodes
workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("ingestion", web_and_ingestion_agent)
workflow.add_node("synthesis", synthesis_and_critique_agent)

# 3. Define the edges (The flow)
# Set the entry point
workflow.set_entry_point("supervisor")

# Supervisor always goes to Ingestion
workflow.add_edge("supervisor", "ingestion")

# Ingestion always goes to Synthesis
workflow.add_edge("ingestion", "synthesis")

# Synthesis uses conditional routing!
workflow.add_conditional_edges(
    "synthesis", # Starting node
    routing_logic, # The function that decides the path
    {
        "continue": "supervisor", # If routing_logic returns "continue", go to supervisor
        "end": END # If routing_logic returns "end", finish the graph
    }
)

# 4. Compile the graph
app = workflow.compile()

In [5]:
# Initialize the state with the user's prompt
initial_state = {
    "original_query": "Analyze the thermal mass properties of locally sourced bamboo versus stabilized earth blocks for a climate-responsive primary school.",
    "search_queries": [],
    "gathered_chunks": [],
    "failed_urls": [],
    "missing_information": [],
    "draft_report": "",
    "is_complete": False
}

# Run the graph
final_state = app.invoke(initial_state)

print("\n=== FINAL REPORT ===")
print(final_state["draft_report"])

---SUPERVISOR AGENT: Generating Queries---
---INGESTION AGENT: Searching and RAG processing---
---SYNTHESIS AGENT: Drafting and Critiquing---
-> Routing: Missing ['compressive strength of structural bamboo']. Looping back to Supervisor.
---SUPERVISOR AGENT: Generating Queries---
---INGESTION AGENT: Searching and RAG processing---
---SYNTHESIS AGENT: Drafting and Critiquing---
-> Routing: Missing ['compressive strength of structural bamboo']. Looping back to Supervisor.
---SUPERVISOR AGENT: Generating Queries---
---INGESTION AGENT: Searching and RAG processing---
---SYNTHESIS AGENT: Drafting and Critiquing---
-> Routing: Missing ['compressive strength of structural bamboo']. Looping back to Supervisor.
---SUPERVISOR AGENT: Generating Queries---
---INGESTION AGENT: Searching and RAG processing---
---SYNTHESIS AGENT: Drafting and Critiquing---
-> Routing: Missing ['compressive strength of structural bamboo']. Looping back to Supervisor.
---SUPERVISOR AGENT: Generating Queries---
---INGEST

GraphRecursionError: Recursion limit of 10007 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [ ]:
2,

In [7]:
a = {1,2,3}
print(type(a))

<class 'set'>
